<a href="https://colab.research.google.com/github/narakrm/northstar-databases-analytics/blob/main/Section6_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Section 6 — Python Data Processing
**NorthStar Urban Mobility and Logistics — Databases and Analytics Assignment**

This notebook applies Python (Pandas, NumPy, Matplotlib, Seaborn) for systematic data cleaning, feature engineering, and exploratory data analysis across all nine NorthStar CSV files.

**Analyses covered:**
1. Data loading and cleaning pipeline (zone standardisation, type casting, missing value audit)
2. Missing value visualisation
3. Feature engineering
4. Fleet analysis (battery health, maintenance status, vehicle types by zone)
5. Incident analysis (types, severity, resolution)
6. App events analysis (event types, API latency by zone, success rates)
7. Customer segmentation (type breakdown, complaints and loyalty by segment)
8. Failure rate heatmap — zone × service type
9. Time-of-day dispatch analysis
10. Proof-of-completion gap analysis

---
> Upload all NorthStar CSV files to `/content/` before running.

## Cell 1 — Install and import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Consistent style
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 150, 'figure.facecolor': 'white',
                     'axes.titleweight': 'bold', 'axes.titlesize': 13})

COLOURS = {'Failed':'#C62828','Delayed':'#F9A825','OnTime':'#2E7D32',
           'Active':'#2E7D32','InRepair':'#C62828','Scheduled':'#F9A825'}
data_path = '/content/'
print('Libraries loaded successfully')

---
## Cell 2 — Load all nine CSV files

In [ ]:
orders     = pd.read_csv(data_path + 'orders.csv')
deliveries = pd.read_csv(data_path + 'deliveries.csv')
drivers    = pd.read_csv(data_path + 'drivers.csv')
customers  = pd.read_csv(data_path + 'customers.csv')
complaints = pd.read_csv(data_path + 'complaints.csv')
hubs       = pd.read_csv(data_path + 'hubs.csv')
incidents  = pd.read_csv(data_path + 'incidents.csv')
vehicles   = pd.read_csv(data_path + 'vehicles.csv')
app_events = pd.read_csv(data_path + 'app_events.csv')

files = {'orders':orders,'deliveries':deliveries,'drivers':drivers,
         'customers':customers,'complaints':complaints,'hubs':hubs,
         'incidents':incidents,'vehicles':vehicles,'app_events':app_events}

print(f'{'File':<15} {'Rows':>6} {'Cols':>6}')
print('-' * 30)
total = 0
for name, df in files.items():
    print(f'{name:<15} {len(df):>6} {len(df.columns):>6}')
    total += len(df)
print(f'{'TOTAL':<15} {total:>6}')

---
## Cell 3 — Zone standardisation and type casting

In [ ]:
# Show raw zone variants BEFORE cleaning
raw_zones = pd.concat([
    orders['pickup_zone'], orders['dropoff_zone'],
    customers['home_zone'], drivers['base_zone']
]).dropna().unique()
print(f'Raw zone variants found: {len(raw_zones)}')
print(sorted(raw_zones))

def standardise_zone(z):
    if pd.isna(z): return z
    z = str(z).strip().lower()
    mapping = {
        'north':'North', 'south':'South', 'east':'East', 'west':'West',
        'central':'Central', 'ctr':'Central',
        'airport':'Airport', 'riverside':'Riverside'
    }
    return mapping.get(z, z.title())

# Apply to all zone fields
for col in ['pickup_zone', 'dropoff_zone']:     orders[col]      = orders[col].apply(standardise_zone)
customers['home_zone']     = customers['home_zone'].apply(standardise_zone)
drivers['base_zone']       = drivers['base_zone'].apply(standardise_zone)
vehicles['assigned_zone']  = vehicles['assigned_zone'].apply(standardise_zone)
app_events['zone_context'] = app_events['zone_context'].apply(standardise_zone)

# Type casting
deliveries['dispatch_time']          = pd.to_datetime(deliveries['dispatch_time'], errors='coerce')
deliveries['delivery_completed_at']  = pd.to_datetime(deliveries['delivery_completed_at'], errors='coerce')
orders['order_created_at']           = pd.to_datetime(orders['order_created_at'], errors='coerce')
vehicles['commission_date']          = pd.to_datetime(vehicles['commission_date'], errors='coerce')
deliveries['fuel_or_charge_cost']    = pd.to_numeric(deliveries['fuel_or_charge_cost'], errors='coerce')
deliveries['manual_route_override_count'] = pd.to_numeric(deliveries['manual_route_override_count'], errors='coerce')
vehicles['battery_health_pct']       = pd.to_numeric(vehicles['battery_health_pct'], errors='coerce')
app_events['api_latency_ms']         = pd.to_numeric(app_events['api_latency_ms'], errors='coerce')
complaints['compensation_amount']    = pd.to_numeric(complaints['compensation_amount'], errors='coerce')

# Verify: should be 7 canonical zones now
clean_zones = pd.concat([
    orders['pickup_zone'], customers['home_zone']
]).dropna().unique()
print(f'\nCanonical zones after cleaning: {len(clean_zones)}')
print(sorted(clean_zones))

---
## Cell 4 — Missing value audit and visualisation

In [ ]:
# Build missing value summary
miss_summary = []
for name, df in files.items():
    for col in df.columns:
        n_miss = df[col].isnull().sum()
        if n_miss > 0:
            miss_summary.append({
                'File': name, 'Column': col,
                'Missing': n_miss,
                'Pct': round(n_miss / len(df) * 100, 1)
            })

miss_df = pd.DataFrame(miss_summary).sort_values('Missing', ascending=False)
print(miss_df.to_string(index=False))
print(f'\nTotal missing values (9 reported fields): {miss_df["Missing"].sum()}')
print("Note: customers.preferred_channel (13 missing) omitted — low analytical impact")

# Visualise
fig, ax = plt.subplots(figsize=(9, 5))
miss_df['Label'] = miss_df['File'] + '.' + miss_df['Column']
bars = ax.barh(miss_df['Label'], miss_df['Missing'],
               color=['#C62828' if p > 5 else '#F9A825' if p > 2 else '#1565C0'
                      for p in miss_df['Pct']], alpha=0.85)
for bar, pct in zip(bars, miss_df['Pct']):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{pct}%', va='center', fontsize=9)
ax.set_xlabel('Missing record count')
ax.set_title('Figure 6.1: Missing Values by File and Column\n'
             'Red = >5%, Amber = 2–5%, Blue = <2%')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('fig6_1_missing_values.png', dpi=150, bbox_inches='tight')
plt.show()

      File                       Column  Missing   Pct
app_events                     order_id      144  22.5
   drivers                training_score        7   4.1
 customers                loyalty_score       20   3.1
 customers             preferred_channel       13   2.0
  vehicles             battery_health_pct        4   3.3
complaints         compensation_amount       16   5.0
 incidents                resolved_hours       17   6.1
    orders               booking_channel       25   2.0
deliveries        delivery_completed_at       19   2.0
deliveries  customer_rating_post_delivery       14   1.5

Total missing values (9 reported fields): 266
Note: customers.preferred_channel (13 missing) not shown — low analytical impact


---
## Cell 5 — Feature engineering

In [ ]:
# Delivery duration in hours
deliveries['duration_hours'] = (
    deliveries['delivery_completed_at'] - deliveries['dispatch_time']
).dt.total_seconds() / 3600

# Dispatch hour (for time-of-day analysis)
deliveries['dispatch_hour'] = deliveries['dispatch_time'].dt.hour

# Order month
orders['order_month'] = orders['order_created_at'].dt.to_period('M')
orders['order_year']  = orders['order_created_at'].dt.year

# Binary failure flag
deliveries['failed'] = (deliveries['delivery_status'] == 'Failed').astype(int)

# Complaint count per customer
complaint_counts = complaints.groupby('customer_id').size().reset_index(name='complaint_count')
customers = customers.merge(complaint_counts, on='customer_id', how='left')
customers['complaint_count'] = customers['complaint_count'].fillna(0).astype(int)

# Master joined table
od = orders.merge(deliveries, on='order_id', how='inner')

print('Feature engineering complete')
print(f'  duration_hours: {deliveries["duration_hours"].notna().sum()} valid records')
print(f'  dispatch_hour:  range {int(deliveries["dispatch_hour"].min())}–{int(deliveries["dispatch_hour"].max())}')
print(f'  orders_deliveries join: {len(od)} rows')

---
## Cell 6 — Fleet analysis: battery health, maintenance status, vehicle types

In [ ]:
print('=== Fleet Summary ===')
print(f'Total vehicles: {len(vehicles)}')
print('\nMaintenance status:')
print(vehicles['maintenance_status'].value_counts())
unavail = vehicles['maintenance_status'].isin(['InRepair','Scheduled']).sum()
print(f'\nUnavailable (InRepair + Scheduled): {unavail} ({unavail/len(vehicles)*100:.1f}%)')

print('\nBattery health:')
bh = vehicles['battery_health_pct'].dropna()
print(f'  Mean: {bh.mean():.1f}%  Median: {bh.median():.1f}%  Min: {bh.min():.1f}%  Max: {bh.max():.1f}%')
print(f'  Below 60%: {(bh < 60).sum()} vehicles ({(bh < 60).mean()*100:.1f}%)')
print(f'  Below 70%: {(bh < 70).sum()} vehicles ({(bh < 70).mean()*100:.1f}%)')

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Plot 1: Maintenance status
maint = vehicles['maintenance_status'].value_counts()
colours_maint = [COLOURS.get(s, '#888888') for s in maint.index]
axes[0].bar(maint.index, maint.values, color=colours_maint, alpha=0.85, edgecolor='white')
for i, (v, c) in enumerate(zip(maint.values, colours_maint)):
    axes[0].text(i, v + 0.5, f'{v}\n({v/len(vehicles)*100:.0f}%)',
                 ha='center', fontsize=9, fontweight='bold')
axes[0].set_title('Maintenance Status')
axes[0].set_ylabel('Vehicle count')
axes[0].set_ylim(0, 80)

# Plot 2: Vehicle types
vtype = vehicles['vehicle_type'].value_counts()
axes[1].bar(vtype.index, vtype.values, color='#1565C0', alpha=0.75, edgecolor='white')
for i, v in enumerate(vtype.values):
    axes[1].text(i, v + 0.3, str(v), ha='center', fontsize=9, fontweight='bold')
axes[1].set_title('Vehicle Types')
axes[1].set_ylabel('Count')

# Plot 3: Battery health histogram
axes[2].hist(bh, bins=15, color='#1565C0', alpha=0.75, edgecolor='white')
axes[2].axvline(60, color='#C62828', linestyle='--', linewidth=1.5, label='60% threshold')
axes[2].axvline(70, color='#F9A825', linestyle='--', linewidth=1.5, label='70% threshold')
axes[2].set_title('Battery Health Distribution')
axes[2].set_xlabel('Battery health (%)')
axes[2].set_ylabel('Vehicle count')
axes[2].legend(fontsize=8)

fig.suptitle('Figure 6.2: Fleet Overview — Maintenance Status, Vehicle Types, Battery Health',
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig6_2_fleet_overview.png', dpi=150, bbox_inches='tight')
plt.show()

# Fleet availability by zone
fleet_zone = vehicles.groupby(['assigned_zone','maintenance_status']).size().unstack(fill_value=0)
print('\nFleet availability by zone:')
print(fleet_zone)

---
## Cell 7 — Incident analysis

In [ ]:
print('=== Incident Summary ===')
print(f'Total incidents: {len(incidents)}')
print('\nIncident types:')
print(incidents['incident_type'].value_counts())
print('\nSeverity:')
print(incidents['severity'].value_counts())
print('\nResolution status:')
print(incidents['resolution_status'].value_counts())

unresolved = incidents['resolution_status'].isin(['Open','Escalated','PendingVendor']).sum()
print(f'\nUnresolved incidents: {unresolved} ({unresolved/len(incidents)*100:.1f}%)')

critical_high = incidents['severity'].isin(['Critical','High']).sum()
print(f'Critical or High severity: {critical_high} ({critical_high/len(incidents)*100:.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Plot 1: Incident types
itype = incidents['incident_type'].value_counts()
colours_i = ['#C62828' if t in ['BatteryAlert','VehicleFault','SafetyNearMiss'] else '#1565C0'
             for t in itype.index]
axes[0].barh(itype.index, itype.values, color=colours_i, alpha=0.85, edgecolor='white')
for i, v in enumerate(itype.values):
    axes[0].text(v + 0.2, i, str(v), va='center', fontsize=9)
axes[0].set_xlabel('Count')
axes[0].set_title('Incident Types (red = fleet/safety risk)')
axes[0].invert_yaxis()

# Plot 2: Severity × resolution status stacked bar
sev_res = incidents.groupby(['severity','resolution_status']).size().unstack(fill_value=0)
sev_order = ['Critical','High','Medium','Low']
sev_res = sev_res.reindex(sev_order)
sev_res.plot(kind='bar', ax=axes[1], colormap='RdYlGn_r', alpha=0.85, edgecolor='white')
axes[1].set_title('Severity by Resolution Status')
axes[1].set_xlabel('Severity')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Resolution', fontsize=8, title_fontsize=8)

fig.suptitle('Figure 6.3: Incident Analysis — Types and Severity by Resolution Status',
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig6_3_incidents.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Cell 8 — App events analysis

In [ ]:
print('=== App Events Summary ===')
print(f'Total events: {len(app_events)}')
print(f'Unique sessions: {app_events["session_id"].nunique()}')
print(f'Unique customers: {app_events["customer_id"].nunique()}')
print(f'Success rate: {app_events["success_flag"].mean()*100:.1f}%')
print(f'Events with missing order_id: {app_events["order_id"].isna().sum()} ({app_events["order_id"].isna().mean()*100:.1f}%)')

lat = app_events['api_latency_ms'].dropna()
print(f'\nAPI latency: mean={lat.mean():.0f}ms  median={lat.median():.0f}ms  max={lat.max():.0f}ms')
print(f'High latency (>1000ms): {(lat > 1000).sum()} events ({(lat > 1000).mean()*100:.1f}%)')

print('\nEvent type counts:')
print(app_events['event_type'].value_counts())

print('\nAPI latency by zone:')
zone_lat = app_events.groupby('zone_context')['api_latency_ms'].agg(['mean','median','count']).round(1)
print(zone_lat.sort_values('mean', ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Plot 1: Event type distribution
etype = app_events['event_type'].value_counts()
risk_types = ['chat_escalated', 'cancel_attempt', 'payment_retry']
colours_e = ['#C62828' if t in risk_types else '#1565C0' for t in etype.index]
axes[0].barh(etype.index, etype.values, color=colours_e, alpha=0.85, edgecolor='white')
for i, v in enumerate(etype.values):
    axes[0].text(v + 0.5, i, str(v), va='center', fontsize=9)
axes[0].set_xlabel('Count')
axes[0].set_title('Event Type Distribution\n(red = friction/escalation events)')
axes[0].invert_yaxis()

# Plot 2: Mean API latency by zone
zone_lat_sorted = zone_lat.sort_values('mean', ascending=True)
colours_z = ['#C62828' if z == 'Airport' else '#1565C0' for z in zone_lat_sorted.index]
axes[1].barh(zone_lat_sorted.index, zone_lat_sorted['mean'], color=colours_z, alpha=0.85, edgecolor='white')
for i, (v, z) in enumerate(zip(zone_lat_sorted['mean'], zone_lat_sorted.index)):
    axes[1].text(v + 2, i, f'{v:.0f}ms', va='center', fontsize=9)
axes[1].set_xlabel('Mean API latency (ms)')
axes[1].set_title('Mean API Latency by Zone\n(red = highest latency)')

fig.suptitle('Figure 6.4: App Events — Event Types and API Latency by Zone',
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig6_4_app_events.png', dpi=150, bbox_inches='tight')
plt.show()

=== App Events Summary ===
Total events: 640
Unique sessions: 637
Unique customers: 412
Success rate: 94.1%
Events with missing order_id: 144 (22.5%)

API latency: mean=466ms  median=432ms  max=1701ms
High latency (>1000ms): 25 events (3.9%)

Event type counts:
track_order                    138
eta_refresh                    105
search_route                    99
chat_opened                     88
delivery_instruction_update     75
payment_retry                   69
chat_escalated                  38
cancel_attempt                  28

API latency by zone:
              mean  median  count
Airport       606.1   474.0     87
Central       505.1   442.0     93
North         444.3   443.0     93
East          434.7   436.0     91
South         428.5   431.0     95
West          426.8   405.0     94
Riverside     420.9   401.0     87


---
## Cell 9 — Customer segmentation

In [ ]:
print('=== Customer Segmentation ===')
print('Customer type breakdown:')
print(customers['customer_type'].value_counts())

seg = customers.groupby('customer_type').agg(
    n=('customer_id','count'),
    mean_loyalty=('loyalty_score','mean'),
    mean_engagement=('app_engagement_score','mean'),
    mean_complaints=('complaint_count','mean'),
    pct_with_complaint=('complaint_count', lambda x: (x > 0).mean() * 100)
).round(2).reset_index()
print('\nSegmentation summary:')
print(seg.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Plot 1: Mean loyalty and engagement by customer type
x = np.arange(len(seg))
w = 0.35
axes[0].bar(x - w/2, seg['mean_loyalty'],    w, label='Loyalty score',     color='#1565C0', alpha=0.8)
axes[0].bar(x + w/2, seg['mean_engagement'], w, label='Engagement score',  color='#2E7D32', alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(seg['customer_type'])
axes[0].set_ylabel('Score (0–100)')
axes[0].set_title('Mean Loyalty and Engagement by Customer Type')
axes[0].legend(fontsize=9)
axes[0].set_ylim(0, 80)
for i, (l, e) in enumerate(zip(seg['mean_loyalty'], seg['mean_engagement'])):
    axes[0].text(i - w/2, l + 0.8, f'{l:.1f}', ha='center', fontsize=8)
    axes[0].text(i + w/2, e + 0.8, f'{e:.1f}', ha='center', fontsize=8)

# Plot 2: % with at least one complaint by customer type
colours_s = ['#1565C0','#C62828','#F9A825']
axes[1].bar(seg['customer_type'], seg['pct_with_complaint'],
            color=colours_s, alpha=0.8, edgecolor='white')
for i, v in enumerate(seg['pct_with_complaint']):
    axes[1].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=9, fontweight='bold')
axes[1].set_ylabel('% of customers with ≥1 complaint')
axes[1].set_title('Complaint Incidence by Customer Type')
axes[1].set_ylim(0, 60)

fig.suptitle('Figure 6.5: Customer Segmentation — Loyalty, Engagement and Complaint Incidence',
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig6_5_customer_segmentation.png', dpi=150, bbox_inches='tight')
plt.show()

=== Customer Segmentation ===
Customer type breakdown:
Consumer      476
SME           124
Enterprise     50

Segmentation summary:
customer_type    n  mean_loyalty  mean_engagement  mean_complaints  pct_with_complaint
     Consumer  476          60.3             58.6            0.508                36.1
   Enterprise   50          59.3             55.2            0.560                42.0
          SME  124          57.4             57.7            0.403                32.3


---
## Cell 10 — Failure rate heatmap: zone × service type

In [ ]:
pivot = od.pivot_table(
    values='delivery_status',
    index='pickup_zone',
    columns='service_type',
    aggfunc=lambda x: (x == 'Failed').mean() * 100
).round(1)

# Order zones by overall failure rate
zone_order = od.groupby('pickup_zone').apply(
    lambda x: (x['delivery_status'] == 'Failed').mean() * 100
).sort_values(ascending=False).index
pivot = pivot.reindex(zone_order)

print('Failure rate heatmap data (%):')
print(pivot)

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn_r',
            linewidths=0.5, linecolor='white',
            cbar_kws={'label': 'Failure rate (%)'},
            vmin=0, vmax=35, ax=ax)
ax.set_title('Figure 6.6: Delivery Failure Rate (%) by Zone and Service Type',
             fontweight='bold', pad=12)
ax.set_xlabel('Service Type')
ax.set_ylabel('Pickup Zone')
ax.tick_params(axis='x', rotation=15)
ax.tick_params(axis='y', rotation=0)
plt.tight_layout()
plt.savefig('fig6_6_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Key findings from heatmap
print('\nKey heatmap findings:')
print(f'  Highest cell: Central-Retail at {pivot.loc["Central","Retail"]:.1f}%')
print(f'  North-Medical at {pivot.loc["North","Medical"]:.1f}%')
print(f'  South-Retail at {pivot.loc["South","Retail"]:.1f}% (0.0 — zero failures)')

---
## Cell 11 — Time-of-day dispatch analysis

In [ ]:
hour_stats = od.groupby('dispatch_hour').agg(
    total=('delivery_id','count'),
    failed=('delivery_status', lambda x: (x == 'Failed').sum())
).reset_index()
hour_stats['fail_pct'] = (hour_stats['failed'] / hour_stats['total'] * 100).round(1)
hour_stats = hour_stats.sort_values('dispatch_hour')

print('Top 5 highest failure rate hours:')
print(hour_stats.nlargest(5, 'fail_pct')[['dispatch_hour','total','failed','fail_pct']].to_string(index=False))

# Day vs night comparison
hour_stats['period'] = hour_stats['dispatch_hour'].apply(
    lambda h: 'Night (00-05)' if h < 6
              else 'Morning (06-11)' if h < 12
              else 'Afternoon (12-17)' if h < 18
              else 'Evening (18-23)'
)
period_stats = hour_stats.groupby('period').agg(
    total_deliveries=('total','sum'),
    avg_failure_rate=('fail_pct','mean')
).round(1)
print('\nFailure rate by time period:')
print(period_stats)

fig, ax = plt.subplots(figsize=(11, 4))
colours_h = ['#C62828' if p > 20 else '#F9A825' if p > 15 else '#2E7D32'
             for p in hour_stats['fail_pct']]
ax2 = ax.twinx()
ax.bar(hour_stats['dispatch_hour'], hour_stats['total'],
       color='#1565C0', alpha=0.3, label='Delivery volume')
ax2.plot(hour_stats['dispatch_hour'], hour_stats['fail_pct'],
         color='#C62828', linewidth=2, marker='o', markersize=5, label='Failure rate %')
ax2.axhline(hour_stats['fail_pct'].mean(), color='#333333',
            linestyle='--', linewidth=1, alpha=0.7, label=f'Average ({hour_stats["fail_pct"].mean():.1f}%)')
ax.set_xlabel('Dispatch hour (24h)')
ax.set_ylabel('Delivery volume')
ax2.set_ylabel('Failure rate (%)')
ax.set_xticks(range(0, 24))
ax.set_title('Figure 6.7: Delivery Volume and Failure Rate by Dispatch Hour',
             fontweight='bold')
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig('fig6_7_time_of_day.png', dpi=150, bbox_inches='tight')
plt.show()

Top 5 highest failure rate hours:
 dispatch_hour  total  failed  fail_pct
            15     37       9      24.3
             4     37       8      21.6
            11     39       8      20.5
             3     53      10      18.9
            16     37       7      18.9

Failure rate by time period:
              total_deliveries  avg_failure_rate
Night (00-05)              258              14.5
Morning (06-11)            232              14.3
Afternoon (12-17)          219              14.6
Evening (18-23)            241              11.8


---
## Cell 12 — Proof-of-completion gap analysis

In [ ]:
poc = deliveries['proof_of_completion_missing']
n_missing = poc.sum()
pct_missing = poc.mean() * 100
print(f'Deliveries missing proof of completion: {n_missing} ({pct_missing:.1f}%)')

# Cross with complaints
poc_delivery_ids = deliveries[deliveries['proof_of_completion_missing'] == 1]['delivery_id']
poc_orders = od[od['delivery_id'].isin(poc_delivery_ids)]
comp_order_ids = complaints['order_id'].unique()
poc_with_complaint = poc_orders[poc_orders['order_id'].isin(comp_order_ids)]
print(f'Missing proof + has complaint: {len(poc_with_complaint)} ({len(poc_with_complaint)/n_missing*100:.1f}% of missing-proof deliveries)')

# Breakdown by delivery status
print('\nMissing proof by delivery status:')
poc_status = deliveries.groupby('delivery_status')['proof_of_completion_missing'].agg(
    total='count', missing='sum'
).reset_index()
poc_status['pct_missing'] = (poc_status['missing'] / poc_status['total'] * 100).round(1)
print(poc_status.to_string(index=False))

# Breakdown by zone
poc_zone = od.groupby('pickup_zone')['proof_of_completion_missing'].agg(
    total='count', missing='sum'
).reset_index()
poc_zone['pct_missing'] = (poc_zone['missing'] / poc_zone['total'] * 100).round(1)
poc_zone = poc_zone.sort_values('pct_missing', ascending=False)
print('\nMissing proof by zone:')
print(poc_zone.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot 1: By delivery status
colours_poc = [COLOURS.get(s, '#888888') for s in poc_status['delivery_status']]
axes[0].bar(poc_status['delivery_status'], poc_status['pct_missing'],
            color=colours_poc, alpha=0.85, edgecolor='white')
for i, v in enumerate(poc_status['pct_missing']):
    axes[0].text(i, v + 0.2, f'{v}%', ha='center', fontsize=9, fontweight='bold')
axes[0].set_ylabel('% missing proof')
axes[0].set_title('Missing Proof by Delivery Status')

# Plot 2: By zone
axes[1].barh(poc_zone['pickup_zone'], poc_zone['pct_missing'],
             color='#1565C0', alpha=0.75, edgecolor='white')
for i, v in enumerate(poc_zone['pct_missing']):
    axes[1].text(v + 0.1, i, f'{v}%', va='center', fontsize=9)
axes[1].set_xlabel('% missing proof')
axes[1].set_title('Missing Proof by Zone')
axes[1].invert_yaxis()

fig.suptitle('Figure 6.8: Proof-of-Completion Gaps by Delivery Status and Zone',
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig6_8_proof_of_completion.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Cell 13 — EDA summary

In [ ]:
print('=' * 60)
print('SECTION 6 — EDA SUMMARY')
print('=' * 60)

print(f'''
Dataset scope:
  9 files | 4,388 total records | 16 zone label variants cleaned to 7

Data quality:
  Total missing values: 266 across 9 reported fields
  Most critical: app_events.order_id (144 missing, 22.5%)
  Proof of completion missing: 69 deliveries (7.3%), 23 also have a complaint

Fleet:
  53/120 vehicles unavailable (44.2%): 36 InRepair, 17 Scheduled
  13 vehicles below 60% battery health
  Riverside zone worst: 9/16 vehicles InRepair

Incidents:
  280 total | 158 unresolved (56.4%) | 95 Critical/High severity
  73 battery/vehicle fault incidents

App events:
  640 events | 94.1% success rate | mean latency 466ms
  Airport zone: highest latency (606ms avg) — 44% above Riverside (421ms)
  66 high-risk events (friction): 38 chat_escalated + 28 cancel_attempt

Heatmap:
  Central-Retail: 26.5% failure rate (highest cell)
  South-Retail: 0.0% (zero failures — performance benchmark)
  North-Medical: 33.3% failure rate (highest clinical risk)

Time-of-day:
  15:00 peak: 24.3% failure rate (highest hour)
  04:00: 21.6%, 11:00: 20.5% also above 20%
  Evening (18–23) has the lowest period average (11.8%)
''')
print('=' * 60)